In [ ]:
from lexguard.ingestion.pdf_loader import load_pdf
from lexguard.ingestion.chunker import build_chunks

pages = load_pdf("data/raw/sample.pdf")

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print(len(chunks))
print(chunks[0])

In [ ]:
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

pages = load_text("../data/raw/sample_policy.txt")

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print("Chunks:", len(chunks))
for c in chunks:
    print("-" * 40)
    print(c.clause_id)
    print(c.chunk_text)

In [ ]:
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

pages = load_text("../data/raw/sample_policy.txt")

chunks = build_chunks(
    document_id="doc_1",
    title="Sample Policy",
    pages=pages
)

print("Chunks:", len(chunks))
for c in chunks:
    print("-" * 50)
    print("clause_id:", c.clause_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text)

In [ ]:
from lexguard.retrieval.bm25 import BM25Retriever

retriever = BM25Retriever(chunks)

results = retriever.query(
    "log retention period",
    top_k=2
)

for r in results:
    print("=" * 50)
    print("section:", r.section_title)
    print("text:", r.chunk_text)

In [ ]:
queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in queries:
    print("\nQUERY:", q)
    results = retriever.query(q, top_k=1)
    print(results[0].chunk_text)

In [ ]:
from lexguard.retrieval.dense import DenseRetriever

dense_retriever = DenseRetriever(chunks)

queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in queries:
    print("\n" + "=" * 60)
    print("QUERY:", q)
    results = dense_retriever.query(q, top_k=2)

    for r in results:
        print("-" * 40)
        print("section:", r.section_title)
        print("text:", r.chunk_text)

In [ ]:
test_queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in test_queries:
    print("\n" + "#" * 70)
    print("QUERY:", q)

    bm25_results = retriever.query(q, top_k=1)
    dense_results = dense_retriever.query(q, top_k=1)

    print("\nBM25:")
    print("section:", bm25_results[0].section_title)
    print("text:", bm25_results[0].chunk_text)

    print("\nDENSE:")
    print("section:", dense_results[0].section_title)
    print("text:", dense_results[0].chunk_text)

In [ ]:
from lexguard.retrieval.hybrid import HybridRetriever

hybrid = HybridRetriever(
    chunks,
    bm25_weight=0.5,
    dense_weight=0.5
)

queries = [
    "log retention period",
    "how long should logs be stored",
    "financial logs retention",
    "log deletion policy",
]

for q in queries:
    print("\n" + "#" * 70)
    print("QUERY:", q)

    results = hybrid.debug_query(q, top_k=2)
    for chunk, score in results:
        print("-" * 40)
        print("score:", round(score, 4))
        print("section:", chunk.section_title)
        print("text:", chunk.chunk_text)